In [170]:
import pandas as pd

portfolio = pd.read_csv("./data/portfolio.csv")
profile = pd.read_csv("./data/profile.csv")
transcript = pd.read_csv("./data/transcript.csv")


In [171]:
# 쓸모없는 인덱스열 삭제
portfolio = portfolio.drop(columns=['Unnamed: 0'], errors='ignore')
profile = profile.drop(columns=['Unnamed: 0'], errors='ignore')

transcript = transcript.rename(columns={'Unnamed: 0': 'index'})  # transcript의 Unnamed컬럼 이름을 index로


In [172]:
# 데이터 타입 date형식으로 변환
profile["became_member_on"] = pd.to_datetime(profile["became_member_on"], format="%Y%m%d")


In [173]:
# channels마다 파생변수 생성

portfolio['web'] = portfolio['channels'].astype(str).str.contains('web').astype(int)
portfolio['email'] = portfolio['channels'].astype(str).str.contains('email').astype(int)
portfolio['mobile'] = portfolio['channels'].astype(str).str.contains('mobile').astype(int)
portfolio['social'] = portfolio['channels'].astype(str).str.contains('social').astype(int)

# 2. channels 컬럼 제거
portfolio = portfolio.drop('channels', axis=1)


In [174]:
import ast

# 1. 딕셔너리처럼 생긴 문자열을 진짜 딕셔너리로 변환
transcript['value'] = transcript['value'].apply(ast.literal_eval)

# 2. 딕셔너리의 키 들을 새로운 컬럼으로 전개
value_df = pd.DataFrame(transcript['value'].tolist())

# 3. transcipt테이블에 이어 붙임
transcript = pd.concat([transcript, value_df], axis=1)

# 4. offer id를 offer_id로 컬럼명 통일
transcript['offer_id'] = transcript['offer_id'].fillna(transcript['offer id'])

# 5. offer id 컬럼은 제거
transcript = transcript.drop('offer id', axis=1)

# 6. value 컬럼 제거
transcript = transcript.drop('value', axis=1)

In [175]:
# transcript 기준으로 profile 데이터를 Left Join
merged_df = pd.merge(transcript, profile, left_on='person', right_on='id', how='left')

# 필요 없는 id 컬럼(person과 중복)은 버리기
merged_df = merged_df.drop(columns='id')


In [176]:
import numpy as np

# 1. gender의 빈칸을 'Unknown'으로 채우기 
merged_df['gender'] = merged_df['gender'].fillna('Unknown')

# 2. age의 118을 진짜 빈칸(NaN)으로 바꿔주기 
merged_df['age'] = merged_df['age'].replace(118, np.nan)

# 3. income은 이미 결측치(NaN) 상태로 비어있기 때문에 아무것도 안 해도 됨

In [178]:
# portfolio 테이블도 병합
all_merge_df = pd.merge(
    merged_df,
    portfolio,
    left_on='offer_id',
    right_on='id',
    how='left'
)

all_merge_df = all_merge_df.drop(columns='id')

# reward열 컬럼 변경
all_merge_df = all_merge_df.rename(columns={
    "reward_x": "transcript_reward",
    "reward_y": "portfolio_reward"
})

In [179]:
# offer_id 이름 변경 (쿠폰명_difficulty_reward_duration)
portfolio['offer_name'] = (
    portfolio['offer_type'] + '_' + 
    portfolio['difficulty'].astype(str) + '_' + 
    portfolio['reward'].astype(str) + '_' + 
    portfolio['duration'].astype(str)
)
# id : key, offer_name : value
offer_name_dict = portfolio.set_index('id')['offer_name'].to_dict()
all_merge_df['offer_id'] = all_merge_df['offer_id'].map(offer_name_dict)

In [180]:
# 무엇을 위해 하는 코드인가? -> informational이 아닌 bogo와 discount 경우만 선택하는 과정

# 조건 1: 쿠폰 타입이 bogo 이거나(in) discount 인 것
cond_offers = all_merge_df['offer_type'].isin(['bogo', 'discount'])

# 조건 2: 이벤트 종류가 transaction(결제) 인 것
cond_transactions = all_merge_df['event'] == 'transaction'

# 위 두 조건 중 하나라도 만족하는(|) 데이터만 쏙 뽑아서 덮어씌우기
target_df = all_merge_df[cond_offers | cond_transactions].copy()

# after: INCLUDE_TRANSACTIONS = True
# after: target_mask = cond_offers | (cond_transactions if INCLUDE_TRANSACTIONS else False)
# after: target_df = all_merge_df[target_mask].copy()  # transaction 포함 여부를 실험적으로 바꿀 때

# 잘 걸러졌는지 눈으로 확인해보기
print(target_df['offer_type'].value_counts(dropna=False))
print(target_df['event'].value_counts(dropna=False))
display(target_df.head())
display(target_df.shape)

offer_type
NaN         138953
bogo         71617
discount     69898
Name: count, dtype: int64
event
transaction        138953
offer received      61042
offer viewed        46894
offer completed     33579
Name: count, dtype: int64


,index,person,event,time,amount,offer_id,transcript_reward,gender,age,became_member_on,income,portfolio_reward,difficulty,duration,offer_type,web,email,mobile,social
0,0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,NaN,bogo_5_5_7,NaN,F,75.0,2017-05-09,100000.0,5.0,5.0,7.0,bogo,1.0,1.0,1.0,0.0
1,1,a03223e636434f42ac4c3df47e8bac43,offer received,0,NaN,discount_20_5_10,NaN,Unknown,NaN,2017-08-04,NaN,5.0,20.0,10.0,discount,1.0,1.0,0.0,0.0
2,2,e2127556f4f64592b11af22de27a7932,offer received,0,NaN,discount_10_2_7,NaN,M,68.0,2018-04-26,70000.0,2.0,10.0,7.0,discount,1.0,1.0,1.0,0.0
3,3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,NaN,discount_10_2_10,NaN,Unknown,NaN,2017-09-25,NaN,2.0,10.0,10.0,discount,1.0,1.0,1.0,1.0
4,4,68617ca6246f4fbc85e91a2a49552598,offer received,0,NaN,bogo_10_10_5,NaN,Unknown,NaN,2017-10-02,NaN,10.0,10.0,5.0,bogo,1.0,1.0,1.0,1.0


(280468, 19)

In [181]:
# person당 offer_id를 하나의 행으로 설정하여, 흩어진 고객 행동의 순서를 보기 편하게 해주는 "피벗테이블 생성 코드"

# 1. 피벗 테이블로 돌릴 '쿠폰 순서' 데이터만 빼내기
offers_df = target_df[target_df['event'] != 'transaction'].copy()

# 2. '순수 거래' 데이터만 빼내기
transactions_df = target_df[target_df['event'] == 'transaction'].copy()

print(f"피벗할 쿠폰 데이터: {len(offers_df)} 줄")
print(f"순수 거래 데이터: {len(transactions_df)} 줄")


피벗할 쿠폰 데이터: 141515 줄
순수 거래 데이터: 138953 줄


In [182]:
# 시간 순으로 정렬 후, 사람&쿠폰별 첫 번째 이벤트만 쏙 뽑아오기
first_events = offers_df.sort_values(['person', 'offer_id', 'time']).groupby(['person', 'offer_id'])['event'].first()

# 그 첫 번째 이벤트가 'received'가 아닌 놈들(유령 데이터)만 필터링
ghost_data = first_events[first_events != 'offer received']

print(f"received 없이 시작하는 유령 세션 수: {len(ghost_data)}개")
if len(ghost_data) > 0:
    display(ghost_data.head())

received 없이 시작하는 유령 세션 수: 0개


In [183]:
# 1. 시간 순서대로 줄 세우기 (시간이 꼬이면 안 되니까 필수)
offers_df = offers_df.sort_values(['person', 'offer_id', 'time'])

# 2. 'received' 이벤트가 등장할 때마다 1, 아니면 0인 깃발(Flag) 만들기
offers_df['is_received'] = (offers_df['event'] == 'offer received').astype(int)

# 3. 사람과 쿠폰 단위로 묶어서, 깃발을 누적해서 더하기
offers_df['offer_cycle'] = offers_df.groupby(['person', 'offer_id'])['is_received'].cumsum()

# 4. 이제 찝찝함 없이 피벗 돌리기 (기준점에 offer_cycle 추가!)
pivot_df = offers_df.pivot_table(
    index=['person', 'offer_id', 'offer_cycle'],  
    columns='event',
    values='time',
    aggfunc='min'
).reset_index()

# 깔끔하게 정리
pivot_df.columns.name = None
pivot_df = pivot_df[['person', 'offer_id', 'offer_cycle', 'offer received', 'offer viewed', 'offer completed']]

display(pivot_df.head())
display(pivot_df.shape)

,person,offer_id,offer_cycle,offer received,offer viewed,offer completed
0,0009655768c64bdeb2e877511632db8f,bogo_5_5_5,1,408.0,456.0,414.0
1,0009655768c64bdeb2e877511632db8f,discount_10_2_10,1,504.0,540.0,528.0
2,0009655768c64bdeb2e877511632db8f,discount_10_2_7,1,576.0,NaN,576.0
3,00116118485d4dfda04fdbaba9a87b5c,bogo_5_5_5,1,168.0,216.0,NaN
4,00116118485d4dfda04fdbaba9a87b5c,bogo_5_5_5,2,576.0,630.0,NaN


(61042, 6)

In [184]:
# 금고(transactions_df)에서 똑같은 사람이 똑같은 시간(time)에 영수증을 2장 이상 긁은 적이 있는지 수색!
multi_receipts = transactions_df.groupby(['person', 'time']).size()

# 2장 이상 긁은 '돌연변이' 케이스만 필터링
mutants = multi_receipts[multi_receipts > 1]

print(f"동일 시간 2회 이상 결제 건수: {len(mutants)}건")
if len(mutants) > 0:
    print("1번 케이스가 존재할 가능성이 있습니다")
else:
    print("휴, 동일 시간에 영수증이 2장인 경우는 없네요. 2번 시나리오입니다")

동일 시간 2회 이상 결제 건수: 0건
휴, 동일 시간에 영수증이 2장인 경우는 없네요. 2번 시나리오입니다


In [144]:
# 1. 원본에서 offer_id와 offer_type 짝꿍 사전 만들기
offer_dict = offers_df[['offer_id', 'offer_type']].drop_duplicates().set_index('offer_id')['offer_type'].to_dict()

# 2. 피벗 테이블의 offer_id를 보고, 임시로 쿠폰 타입(bogo, discount)을 가져오기
temp_offer_type = pivot_df['offer_id'].map(offer_dict)

# 3. [핵심] 기존 숫자였던 'offer_cycle' 컬럼 위에 곧바로 덮어쓰기 
pivot_df['offer_cycle'] = temp_offer_type.str.capitalize() + '_' + pivot_df['offer_cycle'].astype(str)


In [145]:
# 결과 확인
display(pivot_df.head())

,person,offer_id,offer_cycle,offer received,offer viewed,offer completed
0,0009655768c64bdeb2e877511632db8f,bogo_5_5_5,Bogo_1,408.0,456.0,414.0
1,0009655768c64bdeb2e877511632db8f,discount_10_2_10,Discount_1,504.0,540.0,528.0
2,0009655768c64bdeb2e877511632db8f,discount_10_2_7,Discount_1,576.0,NaN,576.0
3,00116118485d4dfda04fdbaba9a87b5c,bogo_5_5_5,Bogo_1,168.0,216.0,NaN
4,00116118485d4dfda04fdbaba9a87b5c,bogo_5_5_5,Bogo_2,576.0,630.0,NaN


In [146]:
# 1. 금고(transactions_df)에서 영수증 알맹이만 꺼내기
transactions_df = transactions_df[['person', 'time', 'amount']]

# 2. 피벗 테이블(pivot_df)에 영수증(receipts) 1:1 도킹하기!
final_df = pivot_df.merge(
    transactions_df,
    left_on=['person', 'offer completed'],  # 도킹 기준: "누구의 / 언제 달성(completed)한 쿠폰인가?"
    right_on=['person', 'time'],      # 도킹 기준: "누가 / 언제(time) 결제했는가?"
    how='left'                       
)

# 3. 끝나고 쓸모없어진 'time' 버리기
final_df = final_df.drop(columns=['time'])

# 4. 최종 결과물 확인
display(final_df.head())
display(final_df.shape)

,person,offer_id,offer_cycle,offer received,offer viewed,offer completed,amount
0,0009655768c64bdeb2e877511632db8f,bogo_5_5_5,Bogo_1,408.0,456.0,414.0,8.57
1,0009655768c64bdeb2e877511632db8f,discount_10_2_10,Discount_1,504.0,540.0,528.0,14.11
2,0009655768c64bdeb2e877511632db8f,discount_10_2_7,Discount_1,576.0,NaN,576.0,10.27
3,00116118485d4dfda04fdbaba9a87b5c,bogo_5_5_5,Bogo_1,168.0,216.0,NaN,NaN
4,00116118485d4dfda04fdbaba9a87b5c,bogo_5_5_5,Bogo_2,576.0,630.0,NaN,NaN


(61042, 7)

In [147]:
# <= 와 <의 행 수 차이 2719


# mask_le = (
#     final_df['offer viewed'].notna() &
#     final_df['offer completed'].notna() &
#     (final_df['offer viewed'] <= final_df['offer completed'])
# )

# mask_lt = (
#     final_df['offer viewed'].notna() &
#     final_df['offer completed'].notna() &
#     (final_df['offer viewed'] < final_df['offer completed'])
# )

# count_le = mask_le.sum()
# count_lt = mask_lt.sum()

# print("<= 개수:", count_le)
# print("< 개수:", count_lt)
# print("차이 나는 행 수:", count_le - count_lt)

# # <=에서는 1인데 <에서는 0인 행만 보기
# diff_df = final_df[mask_le & ~mask_lt]
# display(diff_df.head())


In [148]:
# 1. '진성 전환' 컬럼 만들기
# 조건: viewed 시간이 존재하고, 그 시간이 completed 시간보다 작거나 같아야 함.
# viewed가 빈칸(NaN)인 건들은, 파이썬이 NaN <= 숫자를 비교할 때 자동으로 False(0)로 처리해 주기 때문에 에러 없이 걸러집니다
final_df['is_true_conversion'] = (final_df['offer viewed'] <= final_df['offer completed']).astype(int) 

# 의미를 명확하게 전달하려면 아래 코드로 NaN을 처리해야하지만 위에 코드도 의미는 같아서 괜찮음
# (
#     final_df['offer viewed'].notna() &
#     final_df['offer completed'].notna() &
#     (final_df['offer viewed'] <= final_df['offer completed'])
# ).astype(int)


# 최종 데이터 확인
display(final_df.head(10))

,person,offer_id,offer_cycle,offer received,offer viewed,offer completed,amount,is_true_conversion
0,0009655768c64bdeb2e877511632db8f,bogo_5_5_5,Bogo_1,408.0,456.0,414.0,8.57,0
1,0009655768c64bdeb2e877511632db8f,discount_10_2_10,Discount_1,504.0,540.0,528.0,14.11,0
2,0009655768c64bdeb2e877511632db8f,discount_10_2_7,Discount_1,576.0,NaN,576.0,10.27,0
3,00116118485d4dfda04fdbaba9a87b5c,bogo_5_5_5,Bogo_1,168.0,216.0,NaN,NaN,0
4,00116118485d4dfda04fdbaba9a87b5c,bogo_5_5_5,Bogo_2,576.0,630.0,NaN,NaN,0
5,0011e0d4e6b944f998e987f904e8c1e5,bogo_5_5_7,Bogo_1,504.0,516.0,576.0,22.05,1
6,0011e0d4e6b944f998e987f904e8c1e5,discount_20_5_10,Discount_1,408.0,432.0,576.0,22.05,1
7,0011e0d4e6b944f998e987f904e8c1e5,discount_7_3_7,Discount_1,168.0,186.0,252.0,11.93,1
8,0020c2b971eb4e9188eac86d93036a77,bogo_10_10_5,Bogo_1,408.0,426.0,510.0,17.24,1
9,0020c2b971eb4e9188eac86d93036a77,bogo_10_10_7,Bogo_1,168.0,NaN,NaN,NaN,0


In [149]:
# 프로모션(bogo, discount) 절차를 잘 적용하고 있는 건수는 얼마나 되는가?
print(final_df["is_true_conversion"].sum())

23267


In [150]:
# profit(순수익) 구하기

# portfolio의 'offer_name'을 key, 'reward'를 value
offer_rewards = portfolio.set_index('offer_name')['reward'].to_dict()
final_df['reward_cost'] = final_df['offer_id'].map(offer_rewards)

# offer_id -> offer_name 이름 통일
final_df.rename(columns={'offer_id': 'offer_name'}, inplace=True)

# profit 계산하기! (amount - reward_cost)
final_df['profit'] = final_df['amount'] - final_df['reward_cost']

final_df = final_df[['person', 'offer_name', 'offer_cycle', 'offer received', 'offer viewed', 'offer completed', 'is_true_conversion', 'amount', 'reward_cost', 'profit']]
display(final_df.head(10))

,person,offer_name,offer_cycle,offer received,offer viewed,offer completed,is_true_conversion,amount,reward_cost,profit
0,0009655768c64bdeb2e877511632db8f,bogo_5_5_5,Bogo_1,408.0,456.0,414.0,0,8.57,5,3.57
1,0009655768c64bdeb2e877511632db8f,discount_10_2_10,Discount_1,504.0,540.0,528.0,0,14.11,2,12.11
2,0009655768c64bdeb2e877511632db8f,discount_10_2_7,Discount_1,576.0,NaN,576.0,0,10.27,2,8.27
3,00116118485d4dfda04fdbaba9a87b5c,bogo_5_5_5,Bogo_1,168.0,216.0,NaN,0,NaN,5,NaN
4,00116118485d4dfda04fdbaba9a87b5c,bogo_5_5_5,Bogo_2,576.0,630.0,NaN,0,NaN,5,NaN
5,0011e0d4e6b944f998e987f904e8c1e5,bogo_5_5_7,Bogo_1,504.0,516.0,576.0,1,22.05,5,17.05
6,0011e0d4e6b944f998e987f904e8c1e5,discount_20_5_10,Discount_1,408.0,432.0,576.0,1,22.05,5,17.05
7,0011e0d4e6b944f998e987f904e8c1e5,discount_7_3_7,Discount_1,168.0,186.0,252.0,1,11.93,3,8.93
8,0020c2b971eb4e9188eac86d93036a77,bogo_10_10_5,Bogo_1,408.0,426.0,510.0,1,17.24,10,7.24
9,0020c2b971eb4e9188eac86d93036a77,bogo_10_10_7,Bogo_1,168.0,NaN,NaN,0,NaN,10,NaN


## 지표들 구하기
- 총매출액
- 프로모션 적용시 매출액
- 투자된 reward 금액
- 프로모션 reward 투자 대비 매출액 ROI

In [151]:
# 이번 프로모션 모든 매출의 합(총매출액)
total_revenue = transactions_df['amount'].sum()
print(f"총 매출액: {total_revenue:,}$")

# transactions_df가 실제 결제(transaction)만 모아둔 표이고, 그 안의 amount가 각 결제 금액임

총 매출액: 1,775,451.97$


In [152]:
# 1. transaction을 구분하기 위한 키 생성
# 같은 사람(person) + 같은 결제 완료 시점(offer completed)을 하나의 거래 건으로 묶기 위함
final_df['tx_key'] = final_df['person'].astype(str) + '_' + final_df['offer completed'].astype(str)

# 2. '결제 전 유효 시청' 플래그를 만듬
# offer viewed가 offer completed보다 빠르거나 같아야 "유효한 시청"으로 인정
final_df['is_valid_view'] = final_df['offer viewed'] <= final_df['offer completed']

# 3. 같은 transaction 안에서 우선순위를 정하기 위해 정렬
#  - 먼저 tx_key 기준으로 같은 거래끼리 묶고
#  - 유효 시청(True)을 먼저 올리고
#  - 그 안에서는 더 늦게 본 offer viewed를 우선시함
final_df = final_df.sort_values(
    by=['tx_key', 'is_valid_view', 'offer viewed'],
    ascending=[True, False, False]
)

# 4. 같은 transaction에서 가장 먼저 오는 행만 남기기 위한 flag
# 정렬 후 첫 번째 행만 True, 나머지는 False가 됨
final_df['primary_flag'] = ~final_df.duplicated(subset=['tx_key'], keep='first')

# 5. 최종적으로 프로모션 귀속 대상으로 인정할 행만 표시
#  - 같은 transaction에서 1등(primary_flag)이어야 하고
#  - 결제 전에 본 프로모션(is_valid_view)이어야 함
final_df['valid_attribution_flag'] = final_df['primary_flag'] & final_df['is_valid_view']

# 6. 최종 인정된 행들의 amount만 합쳐서 프로모션 귀속 총매출액 계산
promo_total_revenue = final_df.loc[final_df['valid_attribution_flag'], 'amount'].sum()

print(f"[최종] 프로모션 귀속 총매출액: ${promo_total_revenue:,.2f}")
print(f"[최종] 귀속된 거래 수: {final_df['valid_attribution_flag'].sum():,}")

[최종] 프로모션 귀속 총매출액: $442,993.24
[최종] 귀속된 거래 수: 22,333


In [154]:
# 1) 두 조건의 개수 확인
count_true_conversion = final_df['is_true_conversion'].sum()
count_valid_attr = final_df['valid_attribution_flag'].sum()
diff = count_true_conversion - count_valid_attr

print(f"is_true_conversion 합계: {count_true_conversion:,}")
print(f"valid_attribution_flag 합계: {count_valid_attr:,}")
print(f"차이: {diff:,}")

# 2) is_true_conversion=1 이지만 valid_attribution_flag=0 인 행만 추출
# - last touch로 선택되지 못한 값들 선택
lost_rows = final_df[
    (final_df['is_true_conversion'] == 1) &
    (final_df['valid_attribution_flag'] == 0)
].copy()

print(f"전환으로는 잡혔지만 최종 귀속에서는 빠진 행 수: {len(lost_rows):,}")

# 3) 실제로 934인지 검증
assert len(lost_rows) == diff, "차이 수와 실제 빠진 행 수가 다름"

display(lost_rows.head())


is_true_conversion 합계: 23,267
valid_attribution_flag 합계: 22,333
차이: 934
전환으로는 잡혔지만 최종 귀속에서는 빠진 행 수: 934


,person,offer_name,offer_cycle,offer received,offer viewed,offer completed,is_true_conversion,amount,reward_cost,profit,tx_key,is_valid_view,primary_flag,valid_attribution_flag
6,0011e0d4e6b944f998e987f904e8c1e5,discount_20_5_10,Discount_1,408.0,432.0,576.0,1,22.05,5,17.05,0011e0d4e6b944f998e987f904e8c1e5_576.0,True,False,False
91,00ae03011f9f49b8a4b3e6d416678b0b,bogo_10_10_7,Bogo_2,504.0,534.0,618.0,1,30.83,10,20.83,00ae03011f9f49b8a4b3e6d416678b0b_618.0,True,False,False
157,00cf1bbec83f4a658f8994e556db4633,discount_10_2_10,Discount_2,336.0,438.0,564.0,1,33.28,2,31.28,00cf1bbec83f4a658f8994e556db4633_564.0,True,False,False
277,015c3d28c67e46aa95e9ec97c27220e8,discount_10_2_7,Discount_1,504.0,510.0,618.0,1,22.12,2,20.12,015c3d28c67e46aa95e9ec97c27220e8_618.0,True,False,False
295,01633b71b3a2457aa7d35d8bcc3afb5a,discount_20_5_10,Discount_1,504.0,552.0,600.0,1,24.76,5,19.76,01633b71b3a2457aa7d35d8bcc3afb5a_600.0,True,False,False


In [155]:
# 프로모션 귀속 총매출액
promo_total_revenue = final_df.loc[final_df['valid_attribution_flag'], 'amount'].sum()

# 실제 reward 총지출액 (지출된 총 지출 비용)
total_reward_cost = final_df.loc[final_df['offer completed'].notna(), 'reward_cost'].sum()

# ROI = 프로모션 투자(reward) 대비 수익(프로모션 총매출액)률
roi = (promo_total_revenue - total_reward_cost) / total_reward_cost * 100

print(f"프로모션 귀속 총 매출액: ${promo_total_revenue:,.2f}")
print(f"실제 reward 총지출액: ${total_reward_cost:,.2f}")
print(f"ROI: {roi:.2f}%")
print(f"실제 수익: {promo_total_revenue - total_reward_cost}")


프로모션 귀속 총 매출액: $442,993.24
실제 reward 총지출액: $162,426.00
ROI: 172.74%
실제 수익: 280567.24


In [156]:
# 1. 시스템상 보상(Reward)이 지급된 모든 데이터 모으기 (기준: offer completed)
completed_df = final_df[final_df['offer completed'].notna()]

# 3. [유효 지출] 진짜 마케팅 퍼널(received->viewed->trans->comp)을 제대로 밟은 고객에게 쓴 돈
# (hyeong uk님이 만든 완벽한 플래그를 그대로 사용합니다!)
valid_reward_cost = completed_df[completed_df['valid_attribution_flag']]['reward_cost'].sum()

# 4. [낭비된 지출] 체리피커나 더블카운팅으로 허공에 날린 돈
wasted_reward_cost = total_reward_cost - valid_reward_cost

print(f"💸 [재무 기준] 프로모션 총 리워드 비용: ${total_reward_cost:,.2f}")
print("-" * 60)
print(f"🎯 [마케팅 기준] 광고로 유도된 '유효 지출': ${valid_reward_cost:,.2f} ({valid_reward_cost/total_reward_cost*100:.1f}%)")
print(f"🗑️ [개선 필요] 체리피커 등에게 '낭비된 지출': ${wasted_reward_cost:,.2f} ({wasted_reward_cost/total_reward_cost*100:.1f}%)")

💸 [재무 기준] 프로모션 총 리워드 비용: $162,426.00
------------------------------------------------------------
🎯 [마케팅 기준] 광고로 유도된 '유효 지출': $109,742.00 (67.6%)
🗑️ [개선 필요] 체리피커 등에게 '낭비된 지출': $52,684.00 (32.4%)


여기서부턴 코드 재확인 필요

In [157]:
# 1. 분석을 위해 '순수 프로모션 기여 데이터'만 따로 떼어냅니다.
valid_promo_df = final_df[final_df['valid_attribution_flag']].copy()

print("[스타벅스 프로모션 심층 분석 리포트]\n")

# ==========================================
# 1️⃣ 프로모션 객단가 (AOV) 분석
# ==========================================
# 전체 결제 건의 객단가 vs 프로모션 결제 건의 객단가
total_aov = transactions_df['amount'].mean()
promo_aov = valid_promo_df['amount'].mean()

print("1. 장바구니 크기(AOV) 효과")
print(f"   - 평소 평균 결제액 (자연 매출 객단가): ${total_aov:.2f}")
print(f"   - 쿠폰 사용 시 평균 결제액 (프로모션 객단가): ${promo_aov:.2f}")
if promo_aov > total_aov:
    print(f"인사이트: 쿠폰을 주면 고객들이 평소보다 ${promo_aov - total_aov:.2f} 만큼 더 씁니다! (업셀링 성공)\n")
else:
    print(f"인사이트: 쿠폰을 줘도 결제액이 늘지 않습니다. 구매 조건(허들)을 높여야 합니다.\n")

# ==========================================
# 2️⃣ BOGO vs Discount 맞짱 뜨기 (A/B Test)
# ==========================================
print("2. 프로모션 타입별 성과 (BOGO vs Discount)")
# 타입별로 묶어서 결제 건수와 매출액 합산
type_performance = valid_promo_df.groupby('offer_name').agg(
    결제건수=('tx_key', 'count'),
    기여매출액=('amount', 'sum')
).reset_index()

for index, row in type_performance.iterrows():
    print(f"   - [{row['offer_name'].upper()}] 기여 결제: {row['결제건수']:,}건 / 기여 매출: ${row['기여매출액']:,.2f}")
print("인사이트: 두 프로모션 중 어떤 것이 회사에 더 많은 현금을 가져다주었는지 한눈에 비교됩니다.\n")

# # ==========================================
# # 3️⃣ 퍼널 전환율 (Funnel CVR) - 전체 발송 대비 최종 생존율
# # ==========================================
# total_received = len(all_merge_df[all_merge_df['event'] == 'offer received'])
# total_valid_completed = valid_promo_df['tx_key'].nunique() # 중복 제거된 순수 결제 건수

# print("3. 마케팅 퍼널 최종 전환율")
# print(f"   - 총 발송된 쿠폰 수: {total_received:,}장")
# print(f"   - 최종 순수 기여 결제 수: {total_valid_completed:,}건")
# print(f"   - 최종 전환율 (CVR): {(total_valid_completed / total_received) * 100:.1f}%")
# print("인사이트: 100명에게 쿠폰을 뿌렸을 때, 실질적인 매출로 응답한 진짜 고객의 비율입니다.")

[스타벅스 프로모션 심층 분석 리포트]

1. 장바구니 크기(AOV) 효과
   - 평소 평균 결제액 (자연 매출 객단가): $12.78
   - 쿠폰 사용 시 평균 결제액 (프로모션 객단가): $19.84
인사이트: 쿠폰을 주면 고객들이 평소보다 $7.06 만큼 더 씁니다! (업셀링 성공)

2. 프로모션 타입별 성과 (BOGO vs Discount)
   - [BOGO_10_10_5] 기여 결제: 2,654건 / 기여 매출: $62,602.39
   - [BOGO_10_10_7] 기여 결제: 2,467건 / 기여 매출: $58,204.91
   - [BOGO_5_5_5] 기여 결제: 3,448건 / 기여 매출: $68,251.35
   - [BOGO_5_5_7] 기여 결제: 2,031건 / 기여 매출: $35,512.63
   - [DISCOUNT_10_2_10] 기여 결제: 4,375건 / 기여 매출: $78,095.81
   - [DISCOUNT_10_2_7] 기여 결제: 1,981건 / 기여 매출: $39,576.25
   - [DISCOUNT_20_5_10] 기여 결제: 1,147건 / 기여 매출: $30,280.09
   - [DISCOUNT_7_3_7] 기여 결제: 4,230건 / 기여 매출: $70,469.81
인사이트: 두 프로모션 중 어떤 것이 회사에 더 많은 현금을 가져다주었는지 한눈에 비교됩니다.



# 2번 프로모션별 계산 count/sum은 더블카운팅을 제거한 데이터를 기준으로 계산된 것임.
    - 모두 더하면 22,333/442,989


In [158]:
# BOGO와 Discount 쿠폰 대상 지정
target_offers = ['bogo', 'discount']

# 원천 데이터에서 해당 쿠폰들만 필터링
target_df = all_merge_df[all_merge_df['offer_type'].isin(target_offers)]

# 1단계: 발송(received)된 총 쿠폰 건수 (쿠폰 단위)
step1_received = len(target_df[target_df['event'] == 'offer received'])

# 2단계: 조회(viewed)된 총 쿠폰 건수 (쿠폰 단위)
step2_viewed = len(target_df[target_df['event'] == 'offer viewed'])

# 3단계: 결제 시간 이전에 조회한 기록이 있는 '쿠폰'의 건수 (쿠폰 단위)
# (매출 중복 제거 전의 순수 쿠폰 달성 건수 적용)
step3_completed = len(final_df[final_df['offer viewed'] <= final_df['offer completed']])

# 발송 대비 조회 전환율 및 이탈률 계산
view_rate = (step2_viewed / step1_received) * 100
view_drop_rate = 100 - view_rate

# 조회 대비 결제 전환율 및 이탈률 계산
purchase_rate = (step3_completed / step2_viewed) * 100
purchase_drop_rate = 100 - purchase_rate

# 최종 전환율 (발송 대비 최종 결제) 계산
total_cvr = (step3_completed / step1_received) * 100

# 결과 출력
print("=== BOGO & Discount 퍼널 전환율 및 이탈률 ===")
print(f"1단계 (발송): {step1_received:,}건")
print(f"2단계 (조회): {step2_viewed:,}건 (전환율: {view_rate:.1f}%, 이탈률: {view_drop_rate:.1f}%)")
print(f"3단계 (결제): {step3_completed:,}건 (전환율: {purchase_rate:.1f}%, 이탈률: {purchase_drop_rate:.1f}%)")
print("-" * 40)
print(f"최종 전환율: {total_cvr:.1f}%")

=== BOGO & Discount 퍼널 전환율 및 이탈률 ===
1단계 (발송): 61,042건
2단계 (조회): 46,894건 (전환율: 76.8%, 이탈률: 23.2%)
3단계 (결제): 23,267건 (전환율: 49.6%, 이탈률: 50.4%)
----------------------------------------
최종 전환율: 38.1%


In [159]:
# 쿠폰별 이탈률도 확인
offer_list = sorted(target_df['offer_id'].dropna().unique())

for offer in offer_list:
    # 1) 해당 쿠폰의 received / viewed / completed만 모으기
    offer_raw_df = target_df[target_df['offer_id'] == offer].copy()

    step1_received = len(offer_raw_df[offer_raw_df['event'] == 'offer received'])
    step2_viewed = len(offer_raw_df[offer_raw_df['event'] == 'offer viewed'])

    # 2) final_df에서 해당 쿠폰만 필터링
    offer_final_df = final_df[final_df['offer_name'] == offer].copy()

    # 3) 조회 후 완료된 건수
    step3_completed = len(
        offer_final_df[offer_final_df['offer viewed'] <= offer_final_df['offer completed']]
    )

    # 4) 전환율 / 이탈률
    view_rate = (step2_viewed / step1_received) * 100 if step1_received > 0 else 0
    view_drop_rate = 100 - view_rate

    purchase_rate = (step3_completed / step2_viewed) * 100 if step2_viewed > 0 else 0
    purchase_drop_rate = 100 - purchase_rate

    total_cvr = (step3_completed / step1_received) * 100 if step1_received > 0 else 0

    print(f"=== {offer} 퍼널 전환율 및 이탈률 ===")
    print(f"1단계 (발송): {step1_received:,}건")
    print(f"2단계 (조회): {step2_viewed:,}건 (전환율: {view_rate:.1f}%, 이탈률: {view_drop_rate:.1f}%)")
    print(f"3단계 (결제): {step3_completed:,}건 (전환율: {purchase_rate:.1f}%, 이탈률: {purchase_drop_rate:.1f}%)")
    print(f"최종 전환율: {total_cvr:.1f}%")
    print("-" * 50)


=== bogo_10_10_5 퍼널 전환율 및 이탈률 ===
1단계 (발송): 7,593건
2단계 (조회): 7,298건 (전환율: 96.1%, 이탈률: 3.9%)
3단계 (결제): 2,739건 (전환율: 37.5%, 이탈률: 62.5%)
최종 전환율: 36.1%
--------------------------------------------------
=== bogo_10_10_7 퍼널 전환율 및 이탈률 ===
1단계 (발송): 7,658건
2단계 (조회): 6,716건 (전환율: 87.7%, 이탈률: 12.3%)
3단계 (결제): 2,582건 (전환율: 38.4%, 이탈률: 61.6%)
최종 전환율: 33.7%
--------------------------------------------------
=== bogo_5_5_5 퍼널 전환율 및 이탈률 ===
1단계 (발송): 7,571건
2단계 (조회): 7,264건 (전환율: 95.9%, 이탈률: 4.1%)
3단계 (결제): 3,514건 (전환율: 48.4%, 이탈률: 51.6%)
최종 전환율: 46.4%
--------------------------------------------------
=== bogo_5_5_7 퍼널 전환율 및 이탈률 ===
1단계 (발송): 7,677건
2단계 (조회): 4,171건 (전환율: 54.3%, 이탈률: 45.7%)
3단계 (결제): 2,106건 (전환율: 50.5%, 이탈률: 49.5%)
최종 전환율: 27.4%
--------------------------------------------------
=== discount_10_2_10 퍼널 전환율 및 이탈률 ===
1단계 (발송): 7,597건
2단계 (조회): 7,327건 (전환율: 96.4%, 이탈률: 3.6%)
3단계 (결제): 4,576건 (전환율: 62.5%, 이탈률: 37.5%)
최종 전환율: 60.2%
--------------------------------------------------
===

In [160]:
# 표로 나타내기
offer_list = sorted(target_df['offer_id'].dropna().unique())

rows = []

for offer in offer_list:
    offer_raw_df = target_df[target_df['offer_id'] == offer].copy()
    offer_final_df = final_df[final_df['offer_name'] == offer].copy()

    step1_received = len(offer_raw_df[offer_raw_df['event'] == 'offer received'])
    step2_viewed = len(offer_raw_df[offer_raw_df['event'] == 'offer viewed'])
    step3_completed = len(
        offer_final_df[offer_final_df['offer viewed'] <= offer_final_df['offer completed']]
    )

    view_rate = (step2_viewed / step1_received) * 100 if step1_received > 0 else 0
    view_drop_rate = 100 - view_rate

    purchase_rate = (step3_completed / step2_viewed) * 100 if step2_viewed > 0 else 0
    purchase_drop_rate = 100 - purchase_rate

    total_cvr = (step3_completed / step1_received) * 100 if step1_received > 0 else 0

    rows.append({
        'offer_name': offer,
        'received': step1_received,
        'viewed': step2_viewed,
        'completed': step3_completed,
        'view_rate(%)': round(view_rate, 1),
        'view_drop_rate(%)': round(view_drop_rate, 1),
        'purchase_rate(%)': round(purchase_rate, 1),
        'purchase_drop_rate(%)': round(purchase_drop_rate, 1),
        'total_cvr(%)': round(total_cvr, 1),
    })

offer_funnel_df = pd.DataFrame(rows)


In [161]:
display(offer_funnel_df.sort_values('total_cvr(%)', ascending=False))


,offer_name,received,viewed,completed,view_rate(%),view_drop_rate(%),purchase_rate(%),purchase_drop_rate(%),total_cvr(%)
4,discount_10_2_10,7597,7327,4576,96.4,3.6,62.5,37.5,60.2
7,discount_7_3_7,7646,7337,4348,96.0,4.0,59.3,40.7,56.9
2,bogo_5_5_5,7571,7264,3514,95.9,4.1,48.4,51.6,46.4
0,bogo_10_10_5,7593,7298,2739,96.1,3.9,37.5,62.5,36.1
1,bogo_10_10_7,7658,6716,2582,87.7,12.3,38.4,61.6,33.7
5,discount_10_2_7,7632,4118,2102,54.0,46.0,51.0,49.0,27.5
3,bogo_5_5_7,7677,4171,2106,54.3,45.7,50.5,49.5,27.4
6,discount_20_5_10,7668,2663,1300,34.7,65.3,48.8,51.2,17.0


# received에서 viewed로 넘어갈 때, 채널의 영향이 클 수도 있어보임

In [162]:
# 고객별 전환/이탈률 확인
# person 기준으로 고객 정보만 따로 뽑기
customer_info = merged_df[['person', 'gender', 'age', 'income']].drop_duplicates('person')

# final_df에 고객 정보 다시 붙이기
seg_df = final_df.merge(customer_info, on='person', how='left')


In [163]:
# 1) 공통 분석용 플래그 생성
#    - viewed: offer viewed가 있으면 1
#    - completed: offer completed가 있으면 1
#    - valid_attr: valid_attribution_flag를 숫자형으로 사용

# 조회 여부를 0/1로 만들기
seg_df['viewed_flag'] = seg_df['offer viewed'].notna().astype(int)

# 완료 여부를 0/1로 만들기
seg_df['completed_flag'] = seg_df['offer completed'].notna().astype(int)

# 귀속 거래 여부를 0/1로 다시 한 번 정리
# valid_attribution_flag 와 같은 의미
seg_df['valid_attr_flag'] = seg_df['valid_attribution_flag'].astype(int)


In [164]:
# 2) gender별 분석
#    - 각 성별 그룹에서 발송/조회/완료/귀속건수를 계산
#    - 전환율과 이탈률을 별도 컬럼으로 계산

# gender별로 묶어서 합계 계산
gender_summary = seg_df.groupby('gender').agg(
    total_offers=('person', 'count'),          # 전체 쿠폰 건수
    viewed=('viewed_flag', 'sum'),             # 조회된 건수
    completed=('completed_flag', 'sum'),       # 완료된 건수
    valid_attr=('valid_attr_flag', 'sum')      # 실제 귀속 거래 건수
).reset_index()

# 발송 대비 조회율
gender_summary['view_rate(%)'] = (
    gender_summary['viewed'] / gender_summary['total_offers'] * 100
).round(1)

# 발송 후 이탈률
gender_summary['view_drop_rate(%)'] = (
    100 - gender_summary['view_rate(%)']
).round(1)

# 조회 대비 완료율
gender_summary['completion_rate(%)'] = (
    gender_summary['completed'] / gender_summary['viewed'] * 100
).round(1)

# 조회 후 이탈률
gender_summary['completion_drop_rate(%)'] = (
    100 - gender_summary['completion_rate(%)']
).round(1)

# 보기 좋은 순서로 정리해서 출력
display(
    gender_summary[
        [
            'gender',
            'total_offers',
            'viewed',
            'completed',
            'view_rate(%)',
            'view_drop_rate(%)',
            'completion_rate(%)',
            'completion_drop_rate(%)',
            'valid_attr'
        ]
    ].sort_values('total_offers', ascending=False)
)


,gender,total_offers,viewed,completed,view_rate(%),view_drop_rate(%),completion_rate(%),completion_drop_rate(%),valid_attr
1,M,30562,23012,16210,75.3,24.7,70.4,29.6,11063
0,F,21918,16876,15297,77.0,23.0,90.6,9.4,10018
3,Unknown,7841,6394,1101,81.5,18.5,17.2,82.8,889
2,O,721,612,493,84.9,15.1,80.6,19.4,363


In [165]:
# 3) age 구간 만들기
#    - 너무 세분화된 나이보다 구간으로 보는 것이 해석에 좋다.

# 나이 구간을 생성
seg_df['age_group'] = pd.cut(
    seg_df['age'],
    bins=[0, 20, 30, 40, 50, 60, 70, 120],  # 구간 경계
    labels=['10s', '20s', '30s', '40s', '50s', '60s', '70+'],  # 구간 이름
    right=False                                  # 왼쪽 포함, 오른쪽 미포함
)

# NaN은 Unknown으로 처리
seg_df['age_group'] = seg_df['age_group'].astype('object').fillna('Unknown')

# age 구간별 집계
age_summary = seg_df.groupby('age_group').agg(
    total_offers=('person', 'count'),
    viewed=('viewed_flag', 'sum'),
    completed=('completed_flag', 'sum'),
    valid_attr=('valid_attr_flag', 'sum')
).reset_index()

# 조회율
age_summary['view_rate(%)'] = (
    age_summary['viewed'] / age_summary['total_offers'] * 100
).round(1)

# 발송 후 이탈률
age_summary['view_drop_rate(%)'] = (
    100 - age_summary['view_rate(%)']
).round(1)

# 완료율
age_summary['completion_rate(%)'] = (
    age_summary['completed'] / age_summary['viewed'] * 100
).round(1)

# 조회 후 이탈률
age_summary['completion_drop_rate(%)'] = (
    100 - age_summary['completion_rate(%)']
).round(1)

# 보기 좋게 정렬해서 출력
display(
    age_summary[
        [
            'age_group',
            'total_offers',
            'viewed',
            'completed',
            'view_rate(%)',
            'view_drop_rate(%)',
            'completion_rate(%)',
            'completion_drop_rate(%)',
            'valid_attr'
        ]
    ].sort_values('total_offers', ascending=False)
)


,age_group,total_offers,viewed,completed,view_rate(%),view_drop_rate(%),completion_rate(%),completion_drop_rate(%),valid_attr
4,50s,12692,9704,8161,76.5,23.5,84.1,15.9,5387
5,60s,10780,8330,6826,77.3,22.7,81.9,18.1,4514
6,70+,10254,7842,6603,76.5,23.5,84.2,15.8,4359
3,40s,8241,6561,4788,79.6,20.4,73.0,27.0,3449
7,Unknown,7841,6394,1101,81.5,18.5,17.2,82.8,889
2,30s,5515,4039,2960,73.2,26.8,73.3,26.7,2018
1,20s,4997,3512,2347,70.3,29.7,66.8,33.2,1516
0,10s,722,512,315,70.9,29.1,61.5,38.5,201


In [166]:
# 4) income 구간 만들기
#    - 소득도 구간으로 나눠서 고객 반응 차이를 본다.

# 소득 구간 생성
seg_df['income_group'] = pd.cut(
    seg_df['income'],
    bins=[0, 30000, 50000, 70000, 90000, 120000, float('inf')],
    labels=['0-30k', '30-50k', '50-70k', '70-90k', '90-120k', '120k+'],
    right=False
)

# NaN은 Unknown으로 처리
seg_df['income_group'] = seg_df['income_group'].astype('object').fillna('Unknown')

# income 구간별 집계
income_summary = seg_df.groupby('income_group').agg(
    total_offers=('person', 'count'),
    viewed=('viewed_flag', 'sum'),
    completed=('completed_flag', 'sum'),
    valid_attr=('valid_attr_flag', 'sum')
).reset_index()

# 조회율
income_summary['view_rate(%)'] = (
    income_summary['viewed'] / income_summary['total_offers'] * 100
).round(1)

# 발송 후 이탈률
income_summary['view_drop_rate(%)'] = (
    100 - income_summary['view_rate(%)']
).round(1)

# 완료율
income_summary['completion_rate(%)'] = (
    income_summary['completed'] / income_summary['viewed'] * 100
).round(1)

# 조회 후 이탈률
income_summary['completion_drop_rate(%)'] = (
    100 - income_summary['completion_rate(%)']
).round(1)

# 보기 좋게 정렬해서 출력
display(
    income_summary[
        [
            'income_group',
            'total_offers',
            'viewed',
            'completed',
            'view_rate(%)',
            'view_drop_rate(%)',
            'completion_rate(%)',
            'completion_drop_rate(%)',
            'valid_attr'
        ]
    ].sort_values('total_offers', ascending=False)
)


,income_group,total_offers,viewed,completed,view_rate(%),view_drop_rate(%),completion_rate(%),completion_drop_rate(%),valid_attr
2,50-70k,17851,14151,10379,79.3,20.7,73.3,26.7,7315
1,30-50k,13618,9457,5965,69.4,30.6,63.1,36.9,3953
3,70-90k,13496,10654,9413,78.9,21.1,88.4,11.6,6273
4,90-120k,8186,6203,6204,75.8,24.2,100.0,0.0,3883
5,Unknown,7841,6394,1101,81.5,18.5,17.2,82.8,889
0,120k+,50,35,39,70.0,30.0,111.4,-11.4,20


In [168]:
# # completion_drop_rate이 음수가 가능한가? 코드 재확인
# 일단 gender, age, income을 보고 추가로 가입일도 볼 수 있으면 보는걸로

# age, income 구간통일

# 보고 + 디스카운트/보고별 디스카운트별/프로모션별 세 개 다보는방향으로

# 컬럼구성은 동일하게 total_offers	viewed	completed	view_rate(%)	view_drop_rate(%)	completion_rate(%)	completion_drop_rate(%)	정상흐름 카운트 + 정상흐름 비율


# ----------------------------------------------------------
# 채널별
# 어느 채널에서 전환율이 높은지?

# 각 채널별 전환율  total_offers	viewed	completed	view_rate(%)	view_drop_rate(%)	completion_rate(%)	completion_drop_rate(%)	정상흐름 카운트 + 정상흐름 비율



# 현재 내 final_df = 더블카운팅 처리를 last_touch로 제거한 flag가 있음

In [ ]:
# 채널별로도 한 번 확인


In [ ]:
# 'offer_name' (쿠폰 종류) 별로 묶어서 성과 분석!
promotion_performance = final_df.groupby('offer_name').agg(
    total_sent=('person', 'count'),                        # 몇 명에게 뿌렸나?
    true_conversion_rate=('is_true_conversion', 'mean'),       # 진성 전환율(%)
    avg_amount_per_conv=('amount', 'mean'),                # 전환된 건당 평균 결제액
    avg_profit_per_conv=('profit', 'mean')                 # 전환된 건당 평균 순수익(마진)!
).reset_index()

# 보기 좋게 %로 변환
promotion_performance['true_conversion_rate'] = (promotion_performance['true_conversion_rate'] * 100).round(1).astype(str) + '%'
# 소수점 2자리로 깔끔하게 정리
promotion_performance['avg_amount_per_conv'] = promotion_performance['avg_amount_per_conv'].round(2)
promotion_performance['avg_profit_per_conv'] = promotion_performance['avg_profit_per_conv'].round(2)

# 수익성(avg_profit)이 높은 순서대로 정렬!
promotion_performance = promotion_performance.sort_values('avg_profit_per_conv', ascending=False)

display(promotion_performance)

,offer_name,total_sent,true_conversion_rate,avg_amount_per_conv,avg_profit_per_conv
6,discount_20_5_10,7668,17.0%,25.86,20.86
5,discount_10_2_7,7632,27.5%,20.64,18.64
4,discount_10_2_10,7597,60.2%,18.50,16.50
7,discount_7_3_7,7646,56.9%,17.60,14.60
2,bogo_5_5_5,7571,46.4%,19.51,14.51
1,bogo_10_10_7,7658,33.7%,23.96,13.96
0,bogo_10_10_5,7593,36.1%,23.80,13.80
3,bogo_5_5_7,7677,27.4%,17.91,12.91


In [ ]:
# 총 매출액은 transactions_df -> 중복되지 않음/여기서 구해야함
# true_conversion_rate (진성 전환율)
# avg_amount_per_conv (전환당 평균 결제액 - 객단가)
# avg_profit_per_conv (전환당 평균 순수익)


In [ ]:
# # 완벽한 정상 흐름(received -> viewed -> completed) 필터링
# # 조건: 받은 시간 <= 본 시간 <= 달성한 시간
# true_funnel_mask = (final_df['offer received'] <= final_df['offer viewed']) & (final_df['offer viewed'] <= final_df['offer completed']) # 이것도 아까처럼 (<= / <) 신경써야함!
# best_cases_df = final_df[true_funnel_mask]

# # offer_name별 지표 만들기
# performance_df = best_cases_df.groupby('offer_name').agg(
#     conversion_count=('person', 'count'),      # 이상적인 전환 카운트
#     total_revenue=('amount', 'sum'),           # 총 발생 매출
#     total_profit=('profit', 'sum'),            # 총 순수익
#     avg_profit=('profit', 'mean')              # 건당 평균 순수익 
# ).reset_index()

# performance_df = performance_df.sort_values(by='total_profit', ascending=False)

# display(performance_df)

## 스타벅스 프로모션 성과 측정을 위한 데이터 마트 구축 보고

1. 오프닝: 우리의 목표 (Hook)

"저는 스타벅스 앱 내에 무작위로 쌓여있던 수십만 건의 이벤트 로그를, 우리 프로모션의 진짜 ROI(투자 대비 수익률)를 측정할 수 있는 '분석용 데이터 마트'로 탈바꿈시킨 결과를 공유하고자 합니다."

2. AS-IS: 기존 데이터의 문제점 (Pain Point)

"기존 데이터는 세로로 길게 쌓이는 단순 로그(Log) 형태였고, 핵심 정보인 결제액과 보상액은 value라는 딕셔너리 안에 숨겨져 있었습니다.
이 상태에서는 '어떤 쿠폰이 매출을 얼마나 견인했는지' 연결할 수가 없었습니다. 단순히 '어제 매출 얼마야?'는 알 수 있어도, '이 BOGO 쿠폰 때문에 발생한 매출이 얼마야?'라는 질문에는 대답할 수 없는 맹인이었던 셈이죠."

3. TO-BE: 고객 여정의 평면화 (Solution)

"그래서 저는 이 데이터를 '고객 1명 + 쿠폰 1장' 단위로 가로로 넓게 펼치는 '평면화(Flattening)' 작업을 진행했습니다. 쿠폰을 받고(Received), 보고(Viewed), 결제하는(Completed) 모든 과정을 한 줄의 타임라인으로 만들고, 거기에 영수증 데이터(Amount)를 1:1로 결합했습니다.
왜 이렇게 했을까요? 바로 핵심 KPI를 정확하게 측정하기 위해서입니다."

4. 🎯 우리가 얻게 된 3가지 핵심 KPI (Highlight)

"이 데이터 마트 구축을 통해, 우리는 이제 껍데기뿐인 '단순 달성률'을 버리고 다음과 같은 진짜 비즈니스 지표를 뽑아낼 수 있게 되었습니다."

KPI 1. 진성 전환율 (True Conversion Rate)

의미: "쿠폰을 달성한 전체 고객이 아니라, '광고를 본 뒤에 결제한(Viewed <= Completed)' 진짜 마케팅 성과만 발라냈습니다. 원래 커피를 마시려다 운 좋게 쿠폰을 쓴 '체리피커'의 허수를 제거한 순도 100%의 전환율입니다."

KPI 2. 프로모션별 평균 순수익 (Average Profit per Offer)

의미: "결제액(Amount)에서 우리가 퍼준 혜택(Reward Cost)을 뺀 **진짜 마진(Profit)**을 쿠폰별로 계산했습니다. 이제 '매출은 높은데 적자 나는 쿠폰'과 '전환율은 낮아도 흑자 폭이 큰 알짜 쿠폰'을 완벽하게 구분할 수 있습니다."

KPI 3. 객단가 방어율 (Average Order Value, AOV)

의미: "할인 폭이 큰 쿠폰일수록 고객이 딱 조건만 채우고 이탈하는지, 아니면 추가 결제를 일으켜 객단가(Amount)를 높이는지 비교할 수 있는 토대를 마련했습니다."

5. 🚨 주의사항: 기여도(Attribution)와 더블 카운팅 (Risk Management)

"한 가지 팀원분들께 주의를 당부드리고 싶은 점이 있습니다
고객이 1번 결제해서 BOGO와 할인 쿠폰 2개를 동시에 달성한 경우, 우리는 두 쿠폰 모두에 매출 기여도를 100% 인정했습니다.
따라서 이 데이터는 '어떤 프로모션이 더 강력한가'를 비교(기여도 분석)할 때만 사용해야 하며, 이 표의 수익을 다 더해서 '전사 총매출'로 보고하는 더블 카운팅의 오류를 범해서는 안 됩니다. 전사 총매출은 원본 영수증(Transaction) 데이터로 별도 산출해야 합니다."

6. Next Step 및 마무리

"현재 BOGO와 Discount 쿠폰에 대한 분석 파이프라인은 완성이 되었습니다. 다음 스텝으로는, 달성(Completed) 로그가 아예 남지 않는 '정보성(Informational) 광고'가 고객의 결제에 미친 영향을 '타임 윈도우(Time-window)' 방식으로 추적하는 로직을 추가로 개발할 예정입니다.